# Regresión logística para pedidos tardíos

In [1]:
import sys
sys.path.insert(0, '/repo/src')
sys.path.insert(0, '/repo/lib/sparkstats/src')

import os
os.makedirs('/home/notebooks/mediciones/etapa3/logistica', exist_ok=True)

## Sesión de Spark

La sesión lee desde el catálogo Iceberg `lake` (apunta a `s3a://olist/warehouse`).
Los JARs de Iceberg y el cliente S3A se montan en `/opt/spark/extra-jars`.

In [2]:
from pyspark.sql import SparkSession

JARS = ','.join([
    '/opt/spark/extra-jars/iceberg-spark-runtime-4.1_2.13-1.11.0.jar',
    '/opt/spark/extra-jars/hadoop-aws-3.4.2.jar',
    '/opt/spark/extra-jars/aws-sdk-bundle-v2-2.29.52.jar',
])

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('logistica_pedidos_tardios')
    .config('spark.jars', JARS)
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.sql.catalog.lake', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.lake.type', 'hadoop')
    .config('spark.sql.catalog.lake.warehouse', 's3a://olist/warehouse')
    .config('spark.hadoop.fs.s3a.endpoint', os.environ['RUSTFS_ENDPOINT'])
    .config('spark.hadoop.fs.s3a.access.key', os.environ['AWS_ACCESS_KEY_ID'])
    .config('spark.hadoop.fs.s3a.secret.key', os.environ['AWS_SECRET_ACCESS_KEY'])
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '4g')
    .config('spark.executor.memory', '3g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark:', spark.version)

26/10/21 23:28:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/21 23:28:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark: 4.1.3


## Paso 1. Lectura de silver

Se leen `silver.pedidos` y `silver.envios` fijando el snapshot_id vigente de cada
tabla para garantizar reproducibilidad. El snapshot_id se registra en MLflow.

In [3]:
from modelo.datos import leer_silver

pedidos, envios, snapshots = leer_silver(spark)
print('Snapshots:', snapshots)
print('Pedidos:', pedidos.count(), 'filas')
print('Envios:', envios.count(), 'filas')

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


Snapshots: {'pedidos': 5667379862444125108, 'envios': 2735045448525631629}
Pedidos: 99441 filas
Envios: 100010 filas


In [4]:
# Columnas disponibles en pedidos
print('Columnas de pedidos:')
print([c for c in pedidos.columns])

Columnas de pedidos:
['order_id', 'customer_zip_code_prefix', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 't_aprobacion', 't_despacho', 't_transito', 't_total', 'dias_atraso', 'tarde', 'holgura_promesa', 'dias_prometidos', 'mes_compra', 'fecha_compra', 'en_periodo_util', 'flag_aprobacion_antes_compra', 'flag_despacho_antes_aprobacion', 'flag_entrega_antes_despacho', 'flag_entrega_antes_compra', 'flag_promesa_antes_compra', 'despacho_fuera_plazo', 'exceso_vendedor', 'exceso_transporte', 'customer_unique_id', 'customer_state', 'customer_region', 'ciudad_uf_cliente', 'customer_lat', 'customer_lng', 'geo_confianza_cliente', 'seller_id', 'n_items', 'n_vendedores', 'precio_total', 'flete_total', 'peso_medio_g', 'volumen_medio_cm3', 'categoria_principal', 'flag_peso_invalido', 'payment_type', 'pago_total', 'review_score', 'review_negativa', 'resena_antes_entreg

## Paso 2. Filtrado de población

Se aplican los filtros en orden y se muestra cuántas filas quedan después de cada uno.

**Sesgo de selección:** condicionar a pedidos entregados excluye a los cancelados.
El modelo predice la probabilidad de atraso dado que el pedido se entrega,
no la probabilidad de que no se entregue. Este sesgo es intencional: en producción
la decisión de intervenir se toma al momento de la compra, antes de saber si se entrega.

In [5]:
from modelo.datos import filtrar_poblacion, tabla_filtros

tabla_filtros(pedidos, None, spark).show(truncate=False)

+--------------------+-------+-------------+
|etapa               |n_filas|n_descartadas|
+--------------------+-------+-------------+
|total_original      |0      |99441        |
|entregados_con_fecha|2971   |96470        |
|en_periodo_util     |267    |96203        |
|sin_flags_integridad|1406   |94797        |
|tarde_no_nulo       |0      |94797        |
+--------------------+-------+-------------+



In [6]:
df_poblacion = filtrar_poblacion(pedidos)
print('Población final:', df_poblacion.count(), 'pedidos')
df_poblacion.groupBy('tarde').count().show()

Población final: 94797 pedidos
+-----+-----+
|tarde|count|
+-----+-----+
| true| 6506|
|false|88291|
+-----+-----+



## Paso 3. Envío crítico y variables del pedido

El envío crítico es el que tiene mayor distancia_km dentro del pedido.
Se usa `max` sobre un `struct` para que todos los atributos salgan del mismo envío
(garantiza determinismo: no puede ser que distancia venga de un envío y seller de otro).

In [7]:
from modelo.features import envio_critico, variables_pedido

envios_crit = envio_critico(envios)
print('Envíos críticos:', envios_crit.count())
envios_crit.show(5)

Envíos críticos: 98666


+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+
|            order_id|n_estados_vendedor|distancia_critica_km|   seller_id_critico|seller_state_critico|distancia_incompleta|
+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+
|00010242fe8c5a6d1...|                 1|  301.29334897974627|48436dade18ac8b2b...|                  SP|               false|
|000229ec398224ef6...|                 1|   312.9505535387454|5b51032eddd242adc...|                  MG|               false|
|00042b26cf59d7ce6...|                 1|   646.4479519758663|df560393f3a51e745...|                  PR|               false|
|00048cc3ae777c65d...|                 1|  161.80880331934551|6426d21aca402a131...|                  SP|               false|
|0005f50442cb953dc...|                 1|  33.338200858478245|ba143b05f0110f0dc...|                  SP|              

In [8]:
df_vars = variables_pedido(df_poblacion, envios_crit)
print('Variables del pedido:', df_vars.count())
# Verificar que mes_del_anio, dia_semana y mismo_estado se crearon
df_vars.select('mes_del_anio', 'dia_semana', 'mismo_estado').show(3)

Variables del pedido: 94797


+------------+----------+------------+
|mes_del_anio|dia_semana|mismo_estado|
+------------+----------+------------+
|           4|         4|        true|
|           2|         7|       false|
|          12|         1|        true|
+------------+----------+------------+
only showing top 3 rows


## Paso 4. Variables históricas sin filtrar el futuro

La etiqueta `tarde` solo se conoce cuando el pedido se entrega.
Por eso el historial del vendedor usa solo pedidos **entregados antes de la fecha
de compra** del pedido que se predice.

La implementación usa agregados diarios por clave sobre la fecha de entrega y ventanas
con filtro de rango sobre el número de día (días desde epoch), evitando joins fila a fila.

In [ ]:
# Checkpoint: si ya existe df_features persistido en S3A, lo cargamos y saltamos
# el cálculo pesado de historial_vendedor (~40 min de window functions).
_CHECKPOINT_PATH = "s3a://olist/checkpoints/df_features"
try:
    df_features = spark.read.parquet(_CHECKPOINT_PATH)
    df_features = df_features.cache()
    df_hist = None
    CHECKPOINT_CARGADO = True
    print(f"Checkpoint cargado desde {_CHECKPOINT_PATH}: {df_features.count():,} filas")
except Exception as _e:
    CHECKPOINT_CARGADO = False
    print(f"Sin checkpoint ({_e.__class__.__name__}), calculando desde cero...")

In [ ]:
if not CHECKPOINT_CARGADO:
    from modelo.features import historial_vendedor, margen_promesa
    df_hist = historial_vendedor(df_vars)
    print('Con historial del vendedor:', df_hist.count())
    df_hist.select('seller_id_critico', 'vol_seller_id_critico_90d', 'tasa_seller_id_critico_90d').show(5)

In [ ]:
if not CHECKPOINT_CARGADO:
    df_features = margen_promesa(df_hist)
    df_features = df_features.cache()
    print('Con margen de promesa:', df_features.count())
    df_features.groupBy('nivel_margen').count().show()
    # Guardar checkpoint en S3A para que próximas ejecuciones salten estos ~40 min
    df_features.write.mode("overwrite").parquet(_CHECKPOINT_PATH)
    print(f"Checkpoint guardado en {_CHECKPOINT_PATH}")
    # Releer desde el checkpoint corta el linaje: si no, la búsqueda de hiperparámetros
    # arrastra el plan completo de features en cada ajuste y el driver se queda sin memoria.
    df_features.unpersist()
    df_features = spark.read.parquet(_CHECKPOINT_PATH).cache()
    print('Releído desde checkpoint:', df_features.count())

## Paso 5. Partición temporal

El corte se hace por `fecha_compra`, no aleatoriamente, para respetar la causalidad
temporal. Entrenar con datos futuros y evaluar con datos pasados violaría la premisa
de que las variables históricas solo usan el pasado.

In [11]:
from modelo.config import FECHA_FIN_TRAIN, FECHA_FIN_VAL, FECHA_FIN_TEST
import pyspark.sql.functions as F

train = df_features.filter(F.col('fecha_compra') <= FECHA_FIN_TRAIN)
val   = df_features.filter((F.col('fecha_compra') > FECHA_FIN_TRAIN) & (F.col('fecha_compra') <= FECHA_FIN_VAL))
test  = df_features.filter((F.col('fecha_compra') > FECHA_FIN_VAL) & (F.col('fecha_compra') <= FECHA_FIN_TEST))

# train.cache()
# val.cache()
# test.cache()

for nombre, df in [('Entrenamiento', train), ('Validación', val), ('Prueba', test)]:
    n = df.count()
    tardios = df.filter(F.col('tarde') == True).count()
    print(f'{nombre}: {n:,} pedidos, {tardios:,} tardíos ({tardios/n*100:.1f}%)')

Entrenamiento: 43,293 pedidos, 2,447 tardíos (5.7%)
Validación: 20,589 pedidos, 2,656 tardíos (12.9%)
Prueba: 30,915 pedidos, 1,403 tardíos (4.5%)


In [12]:
# Verificar que la tasa de tardios no cae artificialmente al final del periodo de prueba
test.groupBy('mes_compra').agg(
    F.count('*').alias('n'),
    F.mean(F.col('tarde').cast('double')).alias('tasa_tardios')
).orderBy('mes_compra').show()

+----------+----+--------------------+
|mes_compra|   n|        tasa_tardios|
+----------+----+--------------------+
|   2018-04|6409|0.046965205180215325|
|   2018-05|6681| 0.06570872623858703|
|   2018-06|5973|0.011719403984597355|
|   2018-07|5591| 0.03612949382936863|
|   2018-08|6261| 0.06245008784539211|
+----------+----+--------------------+



## Paso 6. Diagnóstico de variables

Solo con el conjunto de entrenamiento. Los diagnósticos informan las decisiones del
Paso 7 pero no las modifican: las decisiones son fijas y están documentadas.

Si algún diagnóstico contradice una decisión del Paso 7, se muestra la evidencia.

In [13]:
from modelo.diagnostico import (
    diagnostico_nulos,
    diagnostico_categorias_nuevas,
    diagnostico_numericas,
    diagnostico_discretas,
    diagnostico_categoricas,
    diagnostico_redundancia,
    diagnostico_relacion_objetivo,
    NUMERICAS, DISCRETAS, CATEGORICAS,
)

CARPETA = '/home/notebooks/mediciones/etapa3/logistica'

print('=== 6.1 Nulos ===')
x = [c for c in CATEGORICAS if c not in ["tramo_critico", "intraurbana"]]
tabla_nulos = diagnostico_nulos(train, val, test, NUMERICAS + DISCRETAS + x)
tabla_nulos.show(truncate=False)

=== 6.1 Nulos ===
+--------------------------+-------------+----------+------+
|columna                   |entrenamiento|validacion|prueba|
+--------------------------+-------------+----------+------+
|dias_prometidos           |0.0          |0.0       |0.0   |
|margen_promesa            |0.15         |0.0       |0.0   |
|distancia_critica_km      |0.52         |0.45      |0.51  |
|precio_total              |0.0          |0.0       |0.0   |
|flete_total               |0.0          |0.0       |0.0   |
|peso_medio_g              |0.0          |0.0       |0.0   |
|volumen_medio_cm3         |0.0          |0.0       |0.0   |
|diff_pago                 |0.0          |0.0       |0.0   |
|vol_seller_id_critico_90d |8.07         |5.53      |4.43  |
|tasa_seller_id_critico_90d|8.07         |5.53      |4.43  |
|n_items                   |0.0          |0.0       |0.0   |
|n_vendedores              |0.0          |0.0       |0.0   |
|n_estados_vendedor        |0.0          |0.0       |0.0   |
|custo

In [14]:
print('=== 6.1b Categorías nuevas en val/test ===')
cats_nuevas = diagnostico_categorias_nuevas(train, val, test, x)
cats_nuevas.show(truncate=False)

=== 6.1b Categorías nuevas en val/test ===
+-------+--------------------+----------+
|columna|conjunto            |categoria |
+-------+--------------------+----------+
|PI     |seller_state_critico|validacion|
|MA     |seller_state_critico|validacion|
|PI     |seller_state_critico|prueba    |
|MA     |seller_state_critico|prueba    |
+-------+--------------------+----------+



In [15]:
for col in ["n_items", "n_vendedores", "n_estados_vendedor",
            "vol_seller_id_critico_90d", 
            "precio_total", "flete_total", "gmv", "diff_pago", "pago_total"]:
    train =  train.withColumn(col, F.col(col).cast("double"))
    
train.printSchema()                              

root
 |-- order_id: string (nullable = true)
 |-- seller_id_critico: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- t_aprobacion: double (nullable = true)
 |-- t_despacho: double (nullable = true)
 |-- t_transito: double (nullable = true)
 |-- t_total: double (nullable = true)
 |-- dias_atraso: integer (nullable = true)
 |-- tarde: boolean (nullable = true)
 |-- holgura_promesa: integer (nullable = true)
 |-- dias_prometidos: integer (nullable = true)
 |-- mes_compra: string (nullable = true)
 |-- fecha_compra: date (nullable = true)
 |-- en_periodo_util: boolean (n

In [16]:
print('=== 6.2 Numéricas ===')
#resumen_num = diagnostico_numericas(train, CARPETA)
#resumen_num.show(truncate=False)

=== 6.2 Numéricas ===


In [17]:
print('=== 6.3 Discretas ===')
tablas_disc = diagnostico_discretas(train)
for col, tabla in tablas_disc.items():
    print(f'--- {col} ---')
    tabla.show()

=== 6.3 Discretas ===
--- n_items ---
+-------+-----+-------+-------------------+
|n_items|    n|eventos|               tasa|
+-------+-----+-------+-------------------+
|    1.0|38968| 2235.0|0.05735475261753233|
|    2.0| 3327|  171.0|0.05139765554553652|
|    3.0|  588|   23.0| 0.0391156462585034|
|    4.0|  250|   12.0|              0.048|
|    5.0|   75|    3.0|               0.04|
|    6.0|   61|    2.0|0.03278688524590164|
|   21.0|    1|    1.0|                1.0|
|    9.0|    2|    0.0|                0.0|
|    7.0|    6|    0.0|                0.0|
|   14.0|    1|    0.0|                0.0|
|    8.0|    5|    0.0|                0.0|
|   11.0|    1|    0.0|                0.0|
|   10.0|    4|    0.0|                0.0|
|   15.0|    2|    0.0|                0.0|
|   12.0|    1|    0.0|                0.0|
|   20.0|    1|    0.0|                0.0|
+-------+-----+-------+-------------------+

--- n_vendedores ---
+------------+-----+-------+--------------------+
|n_vendedo

In [18]:
print('=== 6.4 Categóricas (entropía y tasa por categoría) ===')
from sparkstats.intervalos import tasa_por_grupo
for col in CATEGORICAS[:5]:  # primeras 5 para brevedad; ver graficos en carpeta
    print(f'--- {col} ---')
    tasa_por_grupo(train, col, 'tarde').show(10)

=== 6.4 Categóricas (entropía y tasa por categoría) ===
--- customer_state ---
+--------------+----+-------+-------------------+-------------------+-------------------+
|customer_state|   n|eventos|               tasa|          wilson_lo|          wilson_hi|
+--------------+----+-------+-------------------+-------------------+-------------------+
|            AL| 198|   41.0|0.20707070707070707|0.15646745513880542|0.26882405495155415|
|            RR|  18|    3.0|0.16666666666666666|0.05836576805861446|0.39222038103915724|
|            MA| 364|   56.0|0.15384615384615385|0.12041326523655091| 0.1945089819513647|
|            SE| 186|   27.0|0.14516129032258066|0.10172017296919553|0.20296279343871423|
|            PB| 247|   30.0| 0.1214574898785425|0.08641679256938647| 0.1680924066707962|
|            PI| 216|   23.0|0.10648148148148148|0.07200809357536724|0.15470737259766953|
|            RJ|5960|  623.0|0.10453020134228187|0.09701593572564891|0.11255393085290379|
|            RN| 228|

In [19]:
print('=== 6.5 Redundancia (V de Cramér y Spearman) ===')
mc, mc2, n_filas_corr = diagnostico_redundancia(train, CARPETA)
print(f'Correlación Spearman calculada con {n_filas_corr:,} filas (sin nulos en numéricas)')
mc.orderBy('cramer_v', ascending=False).show(10)

=== 6.5 Redundancia (V de Cramér y Spearman) ===


Correlación Spearman calculada con 39,596 filas (sin nulos en numéricas)
+--------------------+--------------------+-------------------+
|               col_a|               col_b|           cramer_v|
+--------------------+--------------------+-------------------+
|      customer_state|     customer_region| 0.9336012231960656|
|      customer_state|        mismo_estado|  0.697983450555992|
|     customer_region|        mismo_estado| 0.4215855109570281|
|seller_state_critico|        mismo_estado| 0.2835915468488353|
|seller_state_critico| categoria_principal|0.23388036536341592|
|      customer_state|distancia_incompleta|0.14881381027201285|
| categoria_principal|distancia_incompleta| 0.1334439701614172|
|seller_state_critico|distancia_incompleta|0.10166423268294641|
|     customer_region|distancia_incompleta|0.09362915761626175|
| categoria_principal|        mismo_estado|0.09028055617943422|
+--------------------+--------------------+-------------------+
only showing top 10 rows


In [20]:
print('=== 6.6 Relación con el objetivo (logit empírico) ===')
diagnostico_relacion_objetivo(train, CARPETA)
print('Gráficos guardados en:', CARPETA)

=== 6.6 Relación con el objetivo (logit empírico) ===


26/10/21 23:52:05 WARN QuantileDiscretizer: Some quantiles were identical. Bucketing to 2 buckets as a result.


Gráficos guardados en: /home/notebooks/mediciones/etapa3/logistica


## Paso 7. Decisiones del analista

Decisiones fijas basadas en los diagnósticos del Paso 6.
Si algún diagnóstico contradice una decisión, se muestra la evidencia aquí.

| Variable | Decisión | Motivo |
|---|---|---|
| `customer_region` | Fuera | Función determinista del estado: V de Cramér ~ 1 |
| `gmv` | Fuera | precio_total + flete_total: colineal por definición |
| `pago_total` | Fuera | Muy correlacionado con gmv; queda `diff_pago` |
| `n_vendedores`, `n_estados_vendedor` | Fuera | Queda `multi_vendedor` (binaria) |
| `tramo_critico` | Fuera | La distancia entra continua con `distancia_critica_km` |
| `mes_compra`, `fecha_compra` | Fuera | No son predictoras al momento de la compra |
| `intraurbana`, `mismo_estado` | Candidatas por ablación | Paso 10 |
| `n_items` | Recorte por tope | Elegido según tabla del Paso 6 |

In [21]:
# Verificar decisión de customer_region: debe tener V de Cramér > 0.95 con customer_state
from sparkstats.asociacion import cramer_v
v = cramer_v(train, 'customer_region', 'customer_state')
print(f'V de Cramér customer_region vs customer_state: {v:.4f}')
assert v > 0.9, f'ALERTA: V de Cramér = {v:.4f} < 0.9. Revisar decisión de excluir customer_region'

V de Cramér customer_region vs customer_state: 0.9336


In [25]:
import importlib, modelo.transformadores as tr
importlib.reload(tr)
from modelo.transformadores import construir_pipeline



## Paso 8. Pipeline

El Pipeline incluye todos los pasos que aprenden de los datos dentro del fit,
para que en la búsqueda de hiperparámetros solo se aprenda con entrenamiento.

In [ ]:
import importlib
import modelo.transformadores as tr
importlib.reload(tr)
from modelo.transformadores import construir_pipeline
import pyspark.sql.functions as F

# intraurbana ya no está en BINARIAS_FINALES en el fuente; nada que filtrar
train = train.withColumn("tarde", F.col("tarde").cast("int"))

# Prueba de que el Pipeline se puede ajustar y guardar/cargar dando predicciones idénticas
pipeline_test = construir_pipeline(min_eventos=20, reg_param=0.01, elastic_net_param=0.5)
modelo_test = pipeline_test.fit(train)

pred_orig = modelo_test.transform(val.limit(100)).select('prediction').collect()

# S3A: accesible tanto desde el driver como desde los executors
ruta_modelo_test = 's3a://olist/models/pipeline_test'
modelo_test.write().overwrite().save(ruta_modelo_test)

from pyspark.ml import PipelineModel
modelo_cargado = PipelineModel.load(ruta_modelo_test)
pred_carg = modelo_cargado.transform(val.limit(100)).select('prediction').collect()

assert pred_orig == pred_carg, 'FALLA: predicciones distintas al guardar/cargar el Pipeline'
print('Pipeline: guardar y cargar da predicciones idénticas.')

## Paso 9. Búsqueda de hiperparámetros

Búsqueda aleatoria de 30 combinaciones (configurable en `config.py`).
Cada combinación se registra como un run anidado en MLflow.
La métrica principal es log-loss de validación.

In [ ]:
import mlflow
import os
from modelo.config import EXPERIMENTO_MLFLOW

# Setear tracking URI explícitamente (por si MLFLOW_TRACKING_URI no está en el entorno)
_tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow:5001")
mlflow.set_tracking_uri(_tracking_uri)
print('MLflow tracking URI:', mlflow.get_tracking_uri())
mlflow.set_experiment(EXPERIMENTO_MLFLOW)

In [ ]:
from modelo.entrenar import busqueda_hiperparametros

train.cache()
val.cache()
train.count()
val.count()
df_features.unpersist() 

mejor_comb, historial = busqueda_hiperparametros(train, val, snapshots, [])
print('Mejor combinación:')
for k, v in mejor_comb.items():
    print(f'  {k}: {v}')

In [ ]:
from modelo.graficos import grafico_busqueda
grafico_busqueda(historial, CARPETA)
print(f'Gráfico guardado en {CARPETA}/busqueda_hiperparametros.png')

## Paso 10. Ablación por grupos

Se quita un grupo de variables a la vez y se mide el cambio de log-loss y PR-AUC
en validación. Un grupo que no empeora sale del modelo final.

In [ ]:
from modelo.entrenar import ablacion
from modelo.graficos import grafico_ablacion

ablacion_df = ablacion(mejor_comb, train, val)
ablacion_df.orderBy('delta_log_loss', ascending=False).show(truncate=False)
grafico_ablacion(ablacion_df, CARPETA)

## Paso 11. Referencias para comparar

Se evalúan dos modelos de referencia que el modelo logístico debe superar:

In [ ]:
from modelo.entrenar import referencias

refs = referencias(train, val)
print('Modelo constante (prevalencia):', refs['constante'])
print('Tasa histórica de la ruta:', refs['tasa_ruta'])
print('Prevalencia de entrenamiento:', refs['prevalencia'])

## Paso 12. Modelo final y evaluación en prueba

El modelo final se entrena con entrenamiento + validación y se evalúa una sola vez
en prueba. La prueba no se abre antes de este paso.

In [ ]:
# Variables finales (después de la ablación; intraurbana excluida: no existe en silver.pedidos)
VARIABLES_FINALES = [
    'dias_prometidos', 'margen_promesa', 'distancia_critica_km',
    'precio_total', 'flete_total', 'peso_medio_g', 'volumen_medio_cm3',
    'diff_pago', 'vol_seller_id_critico_90d', 'tasa_seller_id_critico_90d',
    'n_items', 'mes_del_anio', 'dia_semana',
    'customer_state', 'seller_state_critico', 'categoria_principal',
    'payment_type', 'mismo_estado', 'multi_vendedor',
    'flag_peso_invalido', 'distancia_incompleta',
]

# Asegurar que val y test también tienen tarde como int
val  = val.withColumn("tarde",  F.col("tarde").cast("int"))
test = test.withColumn("tarde", F.col("tarde").cast("int"))

In [ ]:
from modelo.entrenar import entrenar_final

modelo_final, run_id = entrenar_final(
    mejor_comb, train, val, snapshots, VARIABLES_FINALES, ablacion_df
)
print('Run ID del modelo final:', run_id)

In [ ]:
from modelo.evaluacion import evaluacion_completa

metricas_prueba = evaluacion_completa(modelo_final, test, CARPETA)
print('\nMétricas de prueba:')
for k, v in metricas_prueba.items():
    if not isinstance(v, (list, dict)):
        print(f'  {k}: {v}')

In [ ]:
# Comparar con las referencias
print('\nComparación con referencias (validación):')
print(f'  Mejor log-loss logística (val): {mejor_comb["log_loss"]:.4f}')
print(f'  Log-loss constante (val):       {refs["constante"]["log_loss"]:.4f}')
print(f'  Log-loss tasa ruta (val):       {refs["tasa_ruta"]["log_loss"]:.4f}')

## Paso 12.5. Estabilidad en prueba

Calibración global y PR-AUC por mes y por región del cliente.
Mayo de 2018 corresponde a la huelga de camioneros en Brasil (18-30 mayo),
que interrumpió la cadena logística.

In [ ]:
from modelo.evaluacion import estabilidad_en_prueba

por_mes, por_region = estabilidad_en_prueba(test, modelo_final, CARPETA)
print('=== Estabilidad por mes ===')
por_mes.show(truncate=False)
print('=== Estabilidad por región ===')
por_region.show(truncate=False)

## Paso 13. Interpretación

Los coeficientes de la regresión logística miden la asociación entre cada variable
y la probabilidad de atraso, **no causalidad**.

Advertencia: los errores estándar son optimistas porque los pedidos de un mismo
vendedor no son independientes (correlación intraclase por vendedor). Un modelo
con errores robustos por clúster daría intervalos de confianza más amplios.

In [ ]:
from modelo.graficos import grafico_coeficientes

grafico_coeficientes(modelo_final, VARIABLES_FINALES, CARPETA)
print('Coeficientes guardados.')

In [ ]:
# GLM binomial sin regularizar para errores estándar
from pyspark.ml.regression import GeneralizedLinearRegression
from pyspark.ml.feature import VectorAssembler

# Nota: si no converge por separación perfecta o colinealidad, se informa abajo
try:
    assembler_glm = VectorAssembler(
        inputCols=['dias_prometidos', 'margen_promesa', 'distancia_critica_km'],
        outputCol='features_glm', handleInvalid='skip'
    )
    train_glm = assembler_glm.transform(train).dropna(subset=['tarde'])
    glm = GeneralizedLinearRegression(
        family='binomial', link='logit',
        featuresCol='features_glm', labelCol='tarde',
        maxIter=100,
    )
    modelo_glm = glm.fit(train_glm)
    print(modelo_glm.summary)
except Exception as e:
    print(f'GLM no convergió: {e}. Posible causa: separación perfecta o colinealidad.')

## Verificaciones finales

Paso 15 del spec: asegurar que no quedan usos de pandas ni de toPandas.

In [ ]:
import subprocess
result = subprocess.run(
    ['grep', '-r', 'toPandas\|import pandas', '/repo/src/', '/home/notebooks/etapa3/'],
    capture_output=True, text=True
)
if result.stdout.strip():
    print('ALERTA: usos encontrados:')
    print(result.stdout)
else:
    print('OK: ningún uso de toPandas ni import pandas en src/ ni notebooks/etapa3/')

In [ ]:
spark.stop()
print('Sesión de Spark cerrada.')